## DESCARGA DE DATOS DE IMPORTACIONES

Vienen de la página del INDEC. Solo bajo los años que voy a necesitar

In [1]:
#En esta celda importo lo que voy a necesitar para la descarga de los datos y para después descomprimirlos y guardarlos en la carpeta correspondiente porque vienen en formato zip.

import time #para que espere entre descargas 
import zipfile #para archivos zip

import requests # para pedir archivos a una página web

from config import CRUDO, GEO, ANIOS #usa cosas de mi archivo config

c:\Users\renat\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
URL_BASE = "https://comex.indec.gob.ar/files/zips/imports_{anio}_Y.zip" #acá dejo un hueco porque después hago un loop que lo integra y necesito bajar datos de 2019 a 2025

CRUDO.mkdir(parents=True, exist_ok=True)

In [3]:
#creo la función descargar que va a bajar cualquier archivo, con reintentos por si falla

def descargar(url, destino, reintentos=3):  # entre paréntesis pongo los parámetros que va a recibir la función, url y destino son obligatorios, reintentos tiene un valor por defecto
    """Descarga un archivo si todavía no está en disco.

    url: dirección del archivo.
    destino: ruta donde guardarlo.
    Devuelve la ruta del archivo, o None si falló después de los reintentos.
    """
    if destino.exists():  # si ya lo tengo, no lo vuelvo a bajar
        return destino
    for intento in range(reintentos): #prueba 3 veces. 0, 1, 2
        try:            #que intente lo que sigue, si no al except
            respuesta = requests.get(url, timeout=60) # pido el archivo; si en 60 segundos no contesta, corto
            respuesta.raise_for_status()              # si la página contestó con un error (ej. 404), lo convierto en error  
            destino.write_bytes(respuesta.content)    # guardo el archivo 
            time.sleep(1)                             # espero 1 segundo para no saturar el servidor
            return destino
        except requests.RequestException as error:      #acá cuando de error va a informar por qué y esperar un tiempo antes de volver a intentar
            print(f"{destino.name}: intento {intento + 1} falló ({error})")
            time.sleep(2 ** intento)
    return None

In [4]:
# Bajo la base de importaciones de cada año y anoto los que fallen

fallidos = []  # lista vacía para anotar los años que no se pudieron bajar
for anio in ANIOS:  # repito para cada año de la lista (2019 a 2025, viene de config.py)
    url = URL_BASE.format(anio=anio)  # lleno el hueco {anio} de la dirección con el año
    destino = CRUDO / f"imports_{anio}_Y.zip"  # armo la ruta donde guardarlo (la / une carpeta y archivo)
    if descargar(url, destino) is None:  # uso la función; si devolvió None, falló...
        fallidos.append(anio)  # ...y agrego ese año a la lista de fallidos
print(f"{len(ANIOS) - len(fallidos)} descargados, {len(fallidos)} fallidos: {fallidos}")  # resumen: cuántos salieron bien y cuáles fallaron

7 descargados, 0 fallidos: []


In [5]:
# Descomprimo cada zip en su propia carpeta

for anio in ANIOS:  # repito para cada año
    zip_path = CRUDO / f"imports_{anio}_Y.zip"  # ruta del zip de ese año
    carpeta = CRUDO / f"imports_{anio}"  # ruta de la carpeta donde lo voy a descomprimir
    if zip_path.exists() and not carpeta.exists():  # si el zip está y todavía no lo descomprimí...
        with zipfile.ZipFile(zip_path) as z:  # ...abro el zip (with lo cierra solo al terminar)
            z.extractall(carpeta)  # saco todos los archivos del zip a la carpeta
    print(anio, sorted(p.name for p in carpeta.iterdir()))  # muestro los archivos que quedaron, en orden alfabético

2019 ['Leame importaciones.docx', 'impoa19.csv', 'itota19.csv']
2020 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2020.xlsx', 'impoa20.csv', 'itota20.csv']
2021 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2021.xlsx', 'impoa21.csv', 'itota21.csv']
2022 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2022.xlsx', 'impoa22.csv', 'itota22.csv']
2023 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2023.xlsx', 'impoa23.csv', 'itota23.csv']
2024 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2024.xlsx', 'impoa24.csv', 'itota24.csv']
2025 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2025.xlsx', 'impoa25.csv', 'itota25.csv']


# Para el mapa

Mapa de países de Natural Earth (escala 1:110 millones, dominio público). Lo bajo con la misma función descargar

In [6]:
# Dirección del mapa de países

URL_MAPA = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
)

GEO.mkdir(parents=True, exist_ok=True)  # creo la carpeta datos/geo si no existe
descargar(URL_MAPA, GEO / "ne_110m_admin_0_countries.geojson")  # reuso la misma función, ahora con el mapa

WindowsPath('datos/geo/ne_110m_admin_0_countries.geojson')